# 12. Закрываем разрыв с AutoML на тех же эмбеддингах

**Контекст:** LightAutoML (ROC 0.815) и AutoGluon (0.806) обходят AutoIntent (linear-голова 0.789, KNN-голова 0.757) на **тех же** e5-фичах. Стекинг из ноутбука 11 дотянул только до 0.805 (soft-voting) — и, что подозрительно, soft-voting там бил настоящий stacking, т.е. мета-уровень был сломан.

**Гипотезы, которые этот ноутбук разводит:**
1. **H1 — дело в классе модели головы.** AutoIntent `classic` не имеет нормально настроенного градиентного бустинга. Если один сильный LightGBM/CatBoost на 1024D ≈ LAMA — разрыв объясняется этим, и вывод для проекта: добавить бустинг-scorer в search space.
2. **H2 — дело в способе блендинга.** Если одиночные головы не дотягивают, но калиброванный взвешенный блендинг / мета-GBM закрывают разрыв — дело в ансамблировании, а не в фичах.
3. **H3 — дело в метрике отбора (F1 vs logloss/AUC).** Проверяем подбором порога под фиксированный ORR: сравниваем модели в равной рабочей точке.

**Что НЕ трогаем:** эмбеддинги. Всё на кэшированных `.npy` из ноутбука 11.

**Методология (как в 11):** 3 сида (42, 123, 456), train = full100k, eval = test (2210), `compute_metrics` идентичен, агрегация mean±std.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score, roc_curve,
)
from scipy.optimize import minimize

try:
    from lightgbm import LGBMClassifier, early_stopping, log_evaluation
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print('⚠️ LightGBM не установлен: pip install lightgbm')

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print('⚠️ CatBoost не установлен')

warnings.filterwarnings('ignore')

# === CONFIG (те же пути, что в ноутбуке 11) ===
BASE = Path('..').resolve()
EMB_CACHE = BASE / 'data/processed/embeddings_cache'
DATA_DIR = BASE / 'data/processed'
OUTPUT_DIR = BASE / 'results/diagnostics'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]
N_FOLDS = 5

# Сначала узнаём ПОТОЛОК на полной мощности. Детерминизм добьёшь потом (thread_count=1 и т.д.).
DETERMINISTIC = False
N_THREADS = 1 if DETERMINISTIC else -1

EVAL_JSONL = DATA_DIR / 'wildjailbreak_eval_binary.jsonl'
EVAL_EMB = EMB_CACHE / 'intfloat_multilingual-e5-large-instruct_test.npy'

# Референсы из metrics_summary_agg.csv (full, mean±std по 3 сидам)
REFS = {
    'REF: LightAutoML':      dict(roc=0.8149, f1=0.8868, orr=0.3746),
    'REF: AutoGluon':        dict(roc=0.8065, f1=0.8746, orr=0.3746),
    'REF: AutoIntent-Linear':dict(roc=0.7890, f1=0.8599, orr=0.3635),
    'REF: AutoIntent-KNN':   dict(roc=0.7569, f1=0.9083, orr=0.6206),
    'REF: Stacking nb11':    dict(roc=0.8051, f1=0.8974, orr=0.4254),
}

print(f'Seeds: {SEEDS} | LightGBM: {LGBM_AVAILABLE} | CatBoost: {CATBOOST_AVAILABLE} | deterministic: {DETERMINISTIC}')

Seeds: [42, 123, 456] | LightGBM: True | CatBoost: True | deterministic: False


## ЭТАП 1: Загрузка данных и метрики (идентично 11)

In [2]:
def load_train_data(seed):
    emb = np.load(EMB_CACHE / f'intfloat_multilingual-e5-large-instruct_full100k_seed{seed}.npy')
    with open(DATA_DIR / f'wildjailbreak_full100k_seed{seed}.json', encoding='utf-8') as f:
        data = json.load(f)
    n_safe = len(data['intents'][0]['utterances'])
    n_jb = len(data['oos_utterances'])
    y = np.array([0]*n_safe + [1]*n_jb)
    return emb, y

def load_eval_data():
    emb = np.load(EVAL_EMB)
    labels = []
    with open(EVAL_JSONL, encoding='utf-8') as f:
        for line in f:
            labels.append(json.loads(line)['label'])
    return emb, np.array(labels)

def compute_metrics(y_true, y_prob, threshold=0.5):
    """Идентично ноутбуку 11."""
    y_pred = (y_prob >= threshold).astype(int)
    safe_mask = y_true == 0
    orr = y_pred[safe_mask].mean() if safe_mask.sum() > 0 else None
    return {
        'roc_auc': roc_auc_score(y_true, y_prob),
        'pr_auc': average_precision_score(y_true, y_prob),
        'f1': f1_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'over_refusal_rate': orr,
    }

def threshold_at_orr(y_true, y_prob, target_orr=0.375):
    """Порог, при котором ORR (FPR на safe) ≈ target. Для сравнения в равной точке."""
    safe = y_prob[y_true == 0]
    thr = np.quantile(safe, 1 - target_orr)  # доля target safe попадёт выше порога
    return float(thr)

eval_emb, eval_labels = load_eval_data()
print(f'Eval: {eval_emb.shape}, jailbreak={int(eval_labels.sum())}, safe={int((eval_labels==0).sum())}')
for s in SEEDS:
    ep = EMB_CACHE / f'intfloat_multilingual-e5-large-instruct_full100k_seed{s}.npy'
    jp = DATA_DIR / f'wildjailbreak_full100k_seed{s}.json'
    print(f'  seed={s}: emb={ep.exists()}, json={jp.exists()}')

Eval: (2210, 1024), jailbreak=2000, safe=210
  seed=42: emb=True, json=True
  seed=123: emb=True, json=True
  seed=456: emb=True, json=True


## ЭТАП 2: Определение голов

Ключевое отличие от 11: **бустинг настроен нормально** — много итераций + ранняя остановка, а не 300 итераций с thread_count=1.

In [3]:
def make_lgbm(seed):
    return LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, num_leaves=63,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        reg_lambda=1.0, class_weight='balanced',
        random_state=seed, n_jobs=N_THREADS, verbose=-1,
    )

def make_catboost(seed):
    return CatBoostClassifier(
        iterations=2000, learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
        auto_class_weights='Balanced', random_seed=seed, verbose=False,
        thread_count=(1 if DETERMINISTIC else -1), task_type='CPU',
        early_stopping_rounds=100,
    )

def fit_head(name, seed, Xtr, ytr, Xval, yval):
    """Обучает голову; для бустинга — с  early stopping по val. Возвращает обученную модель."""
    if name == 'linear':
        m = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=seed)
        m.fit(Xtr, ytr)
    elif name == 'knn':
        m = KNeighborsClassifier(n_neighbors=10, metric='cosine', weights='distance')
        m.fit(Xtr, ytr)
    elif name == 'lgbm':
        m = make_lgbm(seed)
        m.fit(Xtr, ytr, eval_set=[(Xval, yval)],
              callbacks=[early_stopping(100, verbose=False), log_evaluation(0)])
    elif name == 'catboost':
        m = make_catboost(seed)
        m.fit(Xtr, ytr, eval_set=(Xval, yval))
    else:
        raise ValueError(name)
    return m

def predict_proba(m, X):
    return m.predict_proba(X)[:, 1]

HEADS = ['linear', 'knn']
if LGBM_AVAILABLE: HEADS.append('lgbm')
if CATBOOST_AVAILABLE: HEADS.append('catboost')
print('Головы:', HEADS)

Головы: ['linear', 'knn', 'lgbm', 'catboost']


## ЭТАП 3: Главный цикл — OOF + eval предсказания по 3 сидам

Для каждого сида: 5-fold CV → OOF-вероятности (для блендинга) + eval-вероятности (усреднённые по фолдам). Бустинг внутри фолда использует hold-out часть фолда как val для early stopping.

In [4]:
all_single = []          # одиночные головы
oof_store = {}           # {(seed): {head: oof_prob}}
evalp_store = {}         # {(seed): {head: eval_prob}}
ylabels_store = {}       # {(seed): y_train}

for seed in SEEDS:
    print(f'\n{"#"*70}\n# SEED {seed}\n{"#"*70}')
    Xtr_full, ytr_full = load_train_data(seed)
    ylabels_store[seed] = ytr_full
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof = {h: np.zeros(len(ytr_full)) for h in HEADS}
    evalp = {h: np.zeros(len(eval_labels)) for h in HEADS}

    for fi, (tr_idx, va_idx) in enumerate(cv.split(Xtr_full, ytr_full)):
        Xtr, ytr = Xtr_full[tr_idx], ytr_full[tr_idx]
        Xva, yva = Xtr_full[va_idx], ytr_full[va_idx]
        for h in HEADS:
            m = fit_head(h, seed, Xtr, ytr, Xva, yva)
            oof[h][va_idx] = predict_proba(m, Xva)
            evalp[h] += predict_proba(m, eval_emb) / N_FOLDS
        print(f'  fold {fi+1}/{N_FOLDS} done')

    oof_store[seed] = oof
    evalp_store[seed] = evalp
    for h in HEADS:
        met = compute_metrics(eval_labels, evalp[h])
        met.update(seed=seed, model=f'single:{h}')
        all_single.append(met)
        print(f"  {h:<10} eval ROC={met['roc_auc']:.4f} F1={met['f1']:.4f} ORR={met['over_refusal_rate']:.4f}")


######################################################################
# SEED 42
######################################################################
  fold 1/5 done
  fold 2/5 done
  fold 3/5 done
  fold 4/5 done
  fold 5/5 done
  linear     eval ROC=0.7796 F1=0.8502 ORR=0.3619
  knn        eval ROC=0.7658 F1=0.9238 ORR=0.7048
  lgbm       eval ROC=0.8200 F1=0.8915 ORR=0.3762
  catboost   eval ROC=0.8000 F1=0.8816 ORR=0.3714

######################################################################
# SEED 123
######################################################################
  fold 1/5 done
  fold 2/5 done
  fold 3/5 done
  fold 4/5 done
  fold 5/5 done
  linear     eval ROC=0.7826 F1=0.8486 ORR=0.3429
  knn        eval ROC=0.7570 F1=0.9221 ORR=0.6905
  lgbm       eval ROC=0.8206 F1=0.8842 ORR=0.3905
  catboost   eval ROC=0.8001 F1=0.8778 ORR=0.4000

######################################################################
# SEED 456
##################################################

## ЭТАП 4: Блендинги — разводим H1 от H2

Три варианта, все без KNN (он тянет вниз) и с KNN — для сравнения:
1. **soft-voting** (равные веса) — как в 11
2. **calibrated weighted** — isotonic-калибровка каждой головы на OOF + веса, оптимизирующие logloss на OOF (как LAMA-блендер)
3. **meta-LightGBM** на [OOF-вероятности | PCA(эмбеддинг, 32D)] — умный стекинг с доступом к сырому сигналу

In [5]:
def neg_logloss_weights(w, P, y):
    w = np.clip(w, 0, None)
    if w.sum() == 0: return 1e9
    w = w / w.sum()
    p = np.clip(P @ w, 1e-6, 1-1e-6)
    return -(y*np.log(p) + (1-y)*np.log(1-p)).mean()

def optimize_weights(P_oof, y):
    k = P_oof.shape[1]
    res = minimize(neg_logloss_weights, np.ones(k)/k, args=(P_oof, y),
                   method='Nelder-Mead', options={'maxiter': 2000, 'xatol':1e-4})
    w = np.clip(res.x, 0, None); w = w/w.sum()
    return w

def calibrate_oof(oof_prob, y, eval_prob):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(oof_prob, y)
    return iso.predict(oof_prob), iso.predict(eval_prob)

all_blend = []
BLEND_SETS = {
    'no_knn': [h for h in HEADS if h != 'knn'],
    'all':    HEADS,
}

for seed in SEEDS:
    oof = oof_store[seed]; evalp = evalp_store[seed]; y = ylabels_store[seed]
    for set_name, heads in BLEND_SETS.items():
        # 1) soft-voting
        sv = np.mean([evalp[h] for h in heads], axis=0)
        m = compute_metrics(eval_labels, sv); m.update(seed=seed, model=f'soft-voting/{set_name}')
        all_blend.append(m)

        # 2) calibrated weighted
        oof_cal, eval_cal = {}, {}
        for h in heads:
            oc, ec = calibrate_oof(oof[h], y, evalp[h])
            oof_cal[h], eval_cal[h] = oc, ec
        P_oof = np.column_stack([oof_cal[h] for h in heads])
        P_eval = np.column_stack([eval_cal[h] for h in heads])
        w = optimize_weights(P_oof, y)
        cw = P_eval @ w
        m = compute_metrics(eval_labels, cw); m.update(seed=seed, model=f'calib-weighted/{set_name}', weights=dict(zip(heads, w.round(3))))
        all_blend.append(m)

        # 3) meta-LightGBM на [вероятности | PCA эмбеддинга]
        if LGBM_AVAILABLE:
            Xtr_full, _ = load_train_data(seed)
            pca = PCA(n_components=32, random_state=seed).fit(Xtr_full)
            tr_pca = pca.transform(Xtr_full); ev_pca = pca.transform(eval_emb)
            meta_tr = np.column_stack([oof[h] for h in heads] + [tr_pca])
            meta_ev = np.column_stack([evalp[h] for h in heads] + [ev_pca])
            # внутренний split для early stopping меты
            from sklearn.model_selection import train_test_split
            mt, mv, yt, yv = train_test_split(meta_tr, y, test_size=0.2, stratify=y, random_state=seed)
            meta = make_lgbm(seed)
            meta.fit(mt, yt, eval_set=[(mv, yv)], callbacks=[early_stopping(100, verbose=False), log_evaluation(0)])
            mp = meta.predict_proba(meta_ev)[:, 1]
            m = compute_metrics(eval_labels, mp); m.update(seed=seed, model=f'meta-lgbm/{set_name}')
            all_blend.append(m)
    print(f'seed {seed}: блендинги готовы')

seed 42: блендинги готовы
seed 123: блендинги готовы
seed 456: блендинги готовы


## ЭТАП 5: Агрегация mean±std + сравнение с референсами

In [6]:
df = pd.DataFrame(all_single + all_blend)
agg = (df.groupby('model')[['roc_auc','pr_auc','f1','precision','recall','over_refusal_rate']]
         .agg(['mean','std']))
agg.columns = ['_'.join(c) for c in agg.columns]
agg = agg.sort_values('roc_auc_mean', ascending=False)

print('='*100)
print('РЕЗУЛЬТАТЫ (mean±std по 3 сидам, порог 0.5)')
print('='*100)
print(f"{'model':<26}{'ROC-AUC':<16}{'F1':<16}{'ORR':<16}")
print('-'*100)
for name, r in agg.iterrows():
    print(f"{name:<26}{r['roc_auc_mean']:.4f}±{r['roc_auc_std']:.4f}   {r['f1_mean']:.4f}±{r['f1_std']:.4f}   {r['over_refusal_rate_mean']:.4f}±{r['over_refusal_rate_std']:.4f}")
print('-'*100)
for name, r in REFS.items():
    print(f"{name:<26}{r['roc']:.4f}           {r['f1']:.4f}           {r['orr']:.4f}")

agg.to_csv(OUTPUT_DIR / 'nb12_heads_blends_agg.csv')
df.to_csv(OUTPUT_DIR / 'nb12_heads_blends_raw.csv', index=False)
print(f'\n→ сохранено в {OUTPUT_DIR}')

РЕЗУЛЬТАТЫ (mean±std по 3 сидам, порог 0.5)
model                     ROC-AUC         F1              ORR             
----------------------------------------------------------------------------------------------------
meta-lgbm/no_knn          0.8196±0.0078   0.9004±0.0022   0.4000±0.0172
single:lgbm               0.8190±0.0024   0.8885±0.0038   0.3794±0.0099
calib-weighted/no_knn     0.8188±0.0023   0.8932±0.0004   0.3889±0.0099
calib-weighted/all        0.8177±0.0020   0.9022±0.0017   0.4048±0.0126
soft-voting/all           0.8131±0.0010   0.8979±0.0007   0.4175±0.0099
meta-lgbm/all             0.8102±0.0058   0.9102±0.0019   0.4524±0.0312
soft-voting/no_knn        0.8084±0.0003   0.8826±0.0026   0.3683±0.0073
single:catboost           0.7996±0.0007   0.8794±0.0020   0.3794±0.0180
single:linear             0.7833±0.0041   0.8504±0.0020   0.3444±0.0167
single:knn                0.7639±0.0062   0.9237±0.0015   0.6921±0.0120
------------------------------------------------------------

## ЭТАП 6: Сравнение в равной рабочей точке (H3)

Подбираем порог под ORR≈0.375 (как у LAMA) на каждой модели и смотрим F1/recall — чтобы развести 'модель лучше' от 'порог удачнее'.

In [7]:
TARGET_ORR = 0.375
rows = []
# собираем eval-вероятности по моделям из single (по сидам) — берём lgbm и лучший бленд
def fixed_orr_metrics(prob_by_seed):
    out = []
    for seed in SEEDS:
        p = prob_by_seed[seed]
        thr = threshold_at_orr(eval_labels, p, TARGET_ORR)
        out.append(compute_metrics(eval_labels, p, threshold=thr))
    return out

candidates = {}
if LGBM_AVAILABLE:
    candidates['lgbm (single)'] = {s: evalp_store[s]['lgbm'] for s in SEEDS}
candidates['linear (single)'] = {s: evalp_store[s]['linear'] for s in SEEDS}

print(f'Сравнение при ORR≈{TARGET_ORR} (mean±std):')
print(f"{'model':<22}{'F1':<16}{'recall':<16}{'ORR(факт)':<16}")
for name, pbs in candidates.items():
    ms = fixed_orr_metrics(pbs)
    f1m = np.mean([m['f1'] for m in ms]); f1s = np.std([m['f1'] for m in ms])
    rcm = np.mean([m['recall'] for m in ms]); rcs = np.std([m['recall'] for m in ms])
    om = np.mean([m['over_refusal_rate'] for m in ms])
    print(f"{name:<22}{f1m:.4f}±{f1s:.4f}   {rcm:.4f}±{rcs:.4f}   {om:.4f}")
print('\nREF LAMA при ORR 0.375: F1 0.887, recall 0.828')

Сравнение при ORR≈0.375 (mean±std):
model                 F1              recall          ORR(факт)       
lgbm (single)         0.8880±0.0062   0.8302±0.0104   0.3762
linear (single)       0.8610±0.0046   0.7858±0.0073   0.3762

REF LAMA при ORR 0.375: F1 0.887, recall 0.828


### Выводы

### Главный результат: H1 подтверждена

Один настроенный **LightGBM** на тех же e5-эмбеддингах даёт **ROC-AUC 0.819 ± 0.002** —
против LightAutoML 0.815 ± 0.002 и AutoGluon 0.806 ± 0.014. LightGBM выше во всех трёх
сидах (0.820 / 0.821 / 0.816), то есть преимущество устойчиво по направлению. Но разница
средних (0.004) при стандартной ошибке разницы ≈ 0.0028 составляет лишь ~1.4σ — при n=3
это **статистическая ничья с LAMA**, не значимый отрыв. Содержательный вывод: бустинг-голова
догоняет AutoML на агрегате, тогда как classic-головы (linear 0.783, KNN 0.764) отставали
явно. Настоящую разницу между LightGBM и LAMA надо искать на adversarial-срезе (→ nb13),
а не на общем ROC, где её размывают лёгкие vanilla-примеры.

CatBoost дал только 0.800 — заметно ниже LightGBM. На M1 CPU с симметричными деревьями и
плотными 1024D-фичами это ожидаемо. Целевая голова для проекта — **LightGBM**.

### H2: блендинг почти ничего не добавляет

Лучший блендинг (meta-lgbm/no_knn, 0.8196) превзошёл одиночный LightGBM (0.8190) на 0.0006 —
в пределах шума, и со втрое бóльшим разбросом (std 0.0078 против 0.0024). Когда базовая голова
сильная, ансамбль из неё и слабых голов не помогает. Это согласуется с наблюдением из nb11
(soft-voting бил stacking). **Вывод: умный стекинг не нужен — нужна одна правильная голова.**

Все блендинги с KNN (/all) стабильно хуже /no_knn и по ROC, и по ORR (0.42–0.45 против 0.38–0.40):
KNN активно тянет ансамбль вниз.

### H3: разрыв — это качество ранжирования модели, а не выбор порога

При выравнивании в одну рабочую точку (ORR ≈ 0.375):

| Модель   | F1            | recall_jailbreak |
|----------|---------------|------------------|
| LightGBM | 0.888 ± 0.006 | 0.830 ± 0.010    |
| linear   | 0.861 ± 0.005 | 0.786 ± 0.007    |
| LAMA (реф) | 0.887       | 0.828            |

При одинаковом ORR LightGBM ловит на ~4.4 п.п. больше джейлбрейков, чем linear, и точно
воспроизводит рабочую точку LAMA. Значит, разрыв по ORR был **не** артефактом неудачного
порога — linear физически хуже ранжирует, поэтому проигрывает при любом пороге. Бустинг
двигает всю ROC-кривую, а не точку на ней. Идея «подобрать порог поумнее у linear-головы»
закрыта.

### Что это означает для проекта

Не нужны ни новые эмбеддинги, ни скрытый слой джейлбрейк-классификатора, ни умный стекинг.
Нужно добавить настроенный LightGBM-scorer в search space AutoIntent (либо починить
`classic-medium`, чтобы его бустинг не был задушен 300 итерациями и thread_count=1).

### Ограничения и открытые вопросы

- **Отрыв от LAMA статистически незначим (1.4σ при n=3).** На агрегированном ROC это ничья;
  агрегат смазан лёгкими vanilla-примерами (ROC ~0.99 из nb09). Чтобы показать реальное
  преимущество — мерить adversarial-срез отдельно и/или увеличить число сидов. → nb13.
- **ORR ≈ 0.38 остаётся высоким** для гардрейла (~80 ложных отказов на 210 safe).
  Требует калибровки порога под бизнес-ограничение FPR, а не оптимизации F1. → ноутбук 13.
- **Few-shot не проверялся** — здесь бустинг, вероятно, хуже устойчивых голов (linear/prototype);
  выбор головы должен зависеть от размера выборки.
- Числа получены при `DETERMINISTIC=False`. Для финального отчёта пересчитать с
  `DETERMINISTIC=True` для воспроизводимости.